# 12 · Slice-based evaluation with FiftyOne

The evaluator already returns per-slice numbers; **FiftyOne** turns them into a
dataset you can explore and adds *mistakenness* and *hardness*. We build a
FiftyOne dataset from the test split and evaluate it; if FiftyOne isn't
installed, we fall back to the same per-slice report in numpy so the cell still
runs.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from harness.data import make_synthetic_dataset, DatasetSpec, load_split, class_names

DATA = Path("../data/bdd-tiny.lance")
if not DATA.exists():
    make_synthetic_dataset(DATA, DatasetSpec(n=3000, seed=7))
print("dataset:", DATA, "| NOTE: these chapters scale to bdd-small/full; here we")
print("demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.")

dataset: ../data/bdd-tiny.lance | NOTE: these chapters scale to bdd-small/full; here we
demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.


In [2]:
from harness.config import load_config
from harness.train import train
model = train(load_config("base", {"budget": {"max_epochs": 10}}), DATA)
xte, yte, mte = load_split(DATA, "test")
pred = model.predict(xte)
names = list(class_names())

In [3]:
try:
    import fiftyone as fo
    from PIL import Image
    import tempfile, os
    d = tempfile.mkdtemp()
    ds = fo.Dataset(name="bdd_eval", overwrite=True)
    samples = []
    for i in range(len(xte)):
        p = os.path.join(d, f"{i}.png")
        Image.fromarray((xte[i] * 255).astype("uint8")).save(p)
        s = fo.Sample(filepath=p)
        s["weather"] = mte["weather"][i]
        s["gt"] = fo.Classification(label=names[yte[i]])
        s["pred"] = fo.Classification(label=names[pred[i]])
        samples.append(s)
    ds.add_samples(samples)
    results = ds.evaluate_classifications("pred", gt_field="gt", method="simple")
    print("FiftyOne per-class report:"); results.print_report()
    for w in sorted(set(mte["weather"])):
        v = ds.match(fo.ViewField("weather") == w)
        acc = v.match(fo.ViewField("pred.label") == fo.ViewField("gt.label")).count() / v.count()
        print(f"  {w:9s} acc={acc:.3f}")
    print("(fo.launch_app(ds.match_tags('foggy')) opens the worst slice in a browser)")
except Exception as e:
    print("FiftyOne unavailable -> numpy per-slice fallback:", type(e).__name__)
    correct = (pred == yte)
    for w in sorted(set(mte["weather"])):
        sel = mte["weather"] == w
        print(f"  {w:9s} acc={correct[sel].mean():.3f}")

   0% ||----------------|   1/450 [38.4ms elapsed, 17.3s remaining, 26.0 samples/s] 

  81% |█████████████/---| 366/450 [215.9ms elapsed, 49.5ms remaining, 1.7K samples/s] 

 100% |█████████████████| 450/450 [261.6ms elapsed, 0s remaining, 1.7K samples/s]     


FiftyOne per-class report:
              precision    recall  f1-score   support

        dawn       0.95      0.92      0.93       154
     daytime       0.95      0.95      0.95       149
       night       0.97      1.00      0.98       147

    accuracy                           0.96       450
   macro avg       0.96      0.96      0.96       450
weighted avg       0.96      0.96      0.96       450

  clear     acc=1.000
  foggy     acc=0.689
  overcast  acc=1.000
  rainy     acc=0.940
  snowy     acc=0.986
(fo.launch_app(ds.match_tags('foggy')) opens the worst slice in a browser)


FiftyOne's value is *looking* at the fog failures and computing **mistakenness**
(likely-wrong labels) and **hardness** (low-margin samples) — the ranking that
feeds mining at scale (Chapter 13). The scalar score is unchanged; FiftyOne is an
analysis layer, not a second scorer.